In [12]:
# ================================
# IMPORTS
# ================================

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

import json
import time

In [31]:
# ================================
# CONFIGURAÇÃO DO DRIVER
# ================================

# Inicializa o navegador 
driver = webdriver.Chrome()

# Cria um "esperador" para elementos dinâmicos
wait = WebDriverWait(driver, 10)

# ================================
# 1. COLETAR TOP 250 FILMES
# ================================

# Acessa a página do Top 250
driver.get("https://www.imdb.com/chart/top/")

# Espera os elementos carregarem
filmes = wait.until(
    EC.presence_of_all_elements_located((
        By.CSS_SELECTOR,
        "li.ipc-metadata-list-summary-item"
    ))
)

print("Filmes encontrados na página:", len(filmes))


Filmes encontrados na página: 250


In [32]:
# ================================
# EXTRAIR LINKS DOS FILMES
# ================================

movie_links = []

for filme in filmes:
    try:
        # Cada item tem um <a> com o link do filme
        link = filme.find_element(By.TAG_NAME, "a").get_attribute("href")
        
        # Remove parâmetros da URL
        clean_link = link.split("?")[0]
        
        movie_links.append(clean_link)

    except:
        pass

# Remove duplicados (se houver)
movie_links = list(set(movie_links))

print("Total de links coletados:", len(movie_links))

Total de links coletados: 250


In [33]:
# ================================
# 2. FUNÇÃO DE SCRAPING DE FILME
# ================================

def scrape_movie(url):
    """
    Acessa a página de um filme e extrai:
    título, ano, nota, poster, gêneros e diretores
    """

    driver.get(url)

    # espera a página carregar
    time.sleep(3)

    # dicionário base (já trata dados faltantes)
    data = {
        "title": None,
        "year": None,
        "rating": None,
        "poster_url": None,
        "genres": [],
        "directors": [],
        "url": url
    }

    # ----------------------------
    # TÍTULO
    # ----------------------------
    try:
        title = driver.find_element(By.TAG_NAME, "h1")
        data["title"] = title.text.strip()
    except:
        pass

    # ----------------------------
    # ANO DE LANÇAMENTO
    # ----------------------------
    try:
        year = driver.find_element(By.XPATH, '//a[contains(@href,"releaseinfo")]')
        data["year"] = year.text.strip()
    except:
        pass

    # ----------------------------
    # NOTA IMDb
    # ----------------------------
    try:
        rating = driver.find_element(By.CSS_SELECTOR, '[data-testid="hero-rating-bar__aggregate-rating__score"] span')
        data["rating"] = rating.text.strip()
    except:
        pass

    # ----------------------------
    # POSTER (URL DA IMAGEM)
    # ----------------------------
    try:
        poster = driver.find_element(By.CSS_SELECTOR, 'img.ipc-image')
        data["poster_url"] = poster.get_attribute("src")
    except:
        pass

    # ----------------------------
    # GÊNEROS
    # ----------------------------
    try:
       # Focando na classe de texto que você enviou no HTML anterior
        genre_elems = driver.find_elements(By.CSS_SELECTOR, 'a.ipc-chip span.ipc-chip__text')
        # Filtramos para pegar apenas os que estão no topo (evitando recomendações de baixo)
        data["genres"] = [g.text.strip() for g in genre_elems[:5] if g.text]

    except:
        pass

    # ----------------------------
    # DIRETORES
    # ----------------------------
    try:
       # O IMDb agrupa Direção, Roteiro e Elenco em <li> com o data-testid abaixo
        # O primeiro bloco de crédito principal geralmente é o Diretor
        dir_section = driver.find_elements(By.CSS_SELECTOR, 'li[data-testid="title-pc-principal-credit"]')
        if dir_section:
            # Pegamos os links de dentro do primeiro bloco de créditos (Direção)
            dir_links = dir_section[0].find_elements(By.TAG_NAME, 'a')
            data["directors"] = [d.text.strip() for d in dir_links if d.text and "name" in d.get_attribute("href")]

    except:
        pass

    return data


In [ ]:
# ================================
# 3. LOOP DE SCRAPING
# ================================

movie_data = []

# Começa com 50 pra testar (depois pode usar 250)
for i, link in enumerate(movie_links[:250]):
    print(f"Processando {i+1}/250")

    data = scrape_movie(link)

    movie_data.append(data)

    # pausa para evitar bloqueio
    time.sleep(1)


# ================================
# 4. SALVAR EM JSON
# ================================

with open("movies.json", "w", encoding="utf-8") as f:
    json.dump(movie_data, f, indent=4, ensure_ascii=False)

print("Arquivo JSON gerado com sucesso!")

Processando 1/2
Processando 2/2
Arquivo JSON gerado com sucesso!


In [35]:
driver.quit()